In [8]:
import os
import torch
from kokoro import KPipeline
import soundfile as sf
from huggingface_hub import HfFileSystem, snapshot_download
import pandas as pd
from tqdm.notebook import tqdm
import requests
import io
import re
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.11.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# # Kokoro TTS Reader Example

# # 1. Quick check to ensure your NVIDIA GPU is active
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# print(f"Using device: {device}")

# # 2. Initialize the Kokoro Pipeline (using 'a' for American English)
# # The first run will automatically download the 82M model weights (~300MB)
# pipeline = KPipeline(lang_code='a', device=device)

# # 3. Define your custom training string
# practice_text = 'Hello, my name is kokoro. I can read text files out loud to you'

# # 4. Choose a native voice and speed
# # Popular voice options include: 'af_bella', 'af_sarah', 'am_adam', 'am_michael'
# voice_choice = 'af_bella' 
# speed_rate = 1.0

# print(f"Generating audio using voice: {voice_choice}...")

# # 5. Run the generator through your GPU cores
# generator = pipeline(practice_text, voice=voice_choice, speed=speed_rate)

# # Kokoro handles the text in parts; loop through and combine them
# for i, (gs, ps, audio) in enumerate(generator):
#     # Set the save output destination path
#     output_filename = f"kokoro_test_part_{i}.wav"
    
#     # Save the raw audio array directly to disk (Kokoro defaults to a clean 24kHz sample rate)
#     sf.write(output_filename, audio, 24000)
#     print(f"🎉 Success! Practice audio saved to {output_filename}")

In [3]:
# Initialize the Kokoro Pipeline and load voices

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
pipeline = KPipeline(lang_code='a', device=device)
load_voice_func = pipeline.load_voice

Using device: cuda


c:\Users\khang\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:71: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "
c:\Users\khang\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [4]:
# Get list of voices from Kokoro-82M huggingface repository

# 1. Connect to the repository on Hugging Face
fs = HfFileSystem()

# 2. List all files ending in .pt inside the voices folder of the repo
# This queries the live repository, so it always sees the newest files
files = fs.glob("hexgrad/Kokoro-82M/voices/*.pt")

# 3. Extract just the filenames (e.g., "af_bella")
# files will be a list like ['hexgrad/Kokoro-82M/voices/af_bella.pt', ...]
all_available_voice_ids = [f.split('/')[-1].replace('.pt', '') for f in files]

# 4. Convert to your DataFrame
self_updating_df = pd.DataFrame(all_available_voice_ids, columns=['Voice_ID'])

# (Optional) Add your attribute parsing logic here as before
print(f"Detected {len(self_updating_df)} voices live from Hugging Face!")
self_updating_df.head(5)

Detected 54 voices live from Hugging Face!


,Voice_ID
0,af_alloy
1,af_aoede
2,af_bella
3,af_heart
4,af_jessica


In [5]:
# Check for voices downloaded and download missing ones from Kokoro-82M repository

# ==========================================
# PART 1: Identify All Online Voices (Source of Truth)
# ==========================================
fs = HfFileSystem()
try:
    online_files = fs.glob("hexgrad/Kokoro-82M/voices/*.pt")
    online_voices = set(f.split("/")[-1].replace(".pt", "") for f in online_files)
except Exception as e:
    print(f"⚠️ Error connecting to Hugging Face repository: {e}")
    online_voices = set()

# ==========================================
# PART 2: Scan Local Hardware Cache Directory
# ==========================================
user_home = os.path.expanduser("~")
cache_dir = os.path.join(user_home, ".cache", "huggingface", "hub", "models--hexgrad--Kokoro-82M", "snapshots")

cached_voices = set()
voices_path = None

if os.path.exists(cache_dir):
    snapshots = os.listdir(cache_dir)
    if snapshots:
        voices_path = os.path.join(cache_dir, snapshots[0], "voices")
        if os.path.exists(voices_path):
            cached_voices = set(f.replace(".pt", "") for f in os.listdir(voices_path) if f.endswith('.pt'))

# ==========================================
# PART 3: Compare Repositories & Synchronize
# ==========================================
print("-" * 50)
print(f"📊 Current Library Status: {len(cached_voices)} / {len(online_voices)} voices cached locally.")
print("-" * 50)

# Determine discrepancies
missing_voices = sorted(list(online_voices - cached_voices))

if not missing_voices:
    print("✨ All voices match. No voices missing from your local hardware cache.")
else:
    print(f"🔍 Found {len(missing_voices)} missing voice(s) dynamically.")
    print(f"📋 Missing list: {', '.join(missing_voices)}\n")
    print("📥 Starting automated pipeline download synchronization...")
    
    newly_downloaded = []
    load_voice_func = pipeline.load_voice  # Utilizing your initialized notebook pipeline instance
    
    # Loop exclusively through missing items with a clean progress tracker
    for voice_id in tqdm(missing_voices, desc="Syncing Local Environment"):
        try:
            load_voice_func(voice_id)
            newly_downloaded.append(voice_id)
        except Exception as e:
            print(f"   ❌ Failed to pull {voice_id}: {e}")
            
    # Final localized confirmation log
    print("\n" + "-" * 50)
    print(f"🎉 Synchronization complete!")
    if newly_downloaded:
        print(f"🚀 Successfully added {len(newly_downloaded)} new voice model(s) to storage:")
        for name in newly_downloaded:
            print(f"   * {name}")
    print("-" * 50)

--------------------------------------------------
📊 Current Library Status: 54 / 54 voices cached locally.
--------------------------------------------------
✨ All voices match. No voices missing from your local hardware cache.


In [6]:
# List name of voices and their attributes

# ==========================================
# PART 1: Get live files from local/remote hub
# ==========================================
fs = HfFileSystem()
files = fs.glob("hexgrad/Kokoro-82M/voices/*.pt")
all_available_voice_ids = [f.split("/")[-1].replace(".pt", "") for f in files]

processed_voices = []
for voice_id in all_available_voice_ids:
    language, gender = "Other / Custom", "Unknown"

    if voice_id.startswith("af_"):
        language, gender = "American English", "Female"
    elif voice_id.startswith("am_"):
        language, gender = "American English", "Male"
    elif voice_id.startswith("bf_"):
        language, gender = "British English", "Female"
    elif voice_id.startswith("bm_"):
        language, gender = "British English", "Male"
    elif voice_id.startswith("jf_"):
        language, gender = "Japanese", "Female"
    elif voice_id.startswith("jm_"):
        language, gender = "Japanese", "Male"
    elif voice_id.startswith("zf_"):
        language, gender = "Chinese", "Female"
    elif voice_id.startswith("zm_"):
        language, gender = "Chinese", "Male"
    elif voice_id.startswith("ff_"):
        language, gender = "French", "Female"
    elif voice_id.startswith("fm_"):
        language, gender = "French", "Male"
    elif voice_id.startswith("hf_"):
        language, gender = "Hindi", "Female"
    elif voice_id.startswith("hm_"):
        language, gender = "Hindi", "Male"
    elif voice_id.startswith("if_"):
        language, gender = "Italian", "Female"
    elif voice_id.startswith("im_"):
        language, gender = "Italian", "Male"
    elif voice_id.startswith("pf_"):
        language, gender = "Brazilian Portuguese", "Female"
    elif voice_id.startswith("pm_"):
        language, gender = "Brazilian Portuguese", "Male"
    elif voice_id.startswith("ef_"):
        language, gender = "Spanish", "Female"
    elif voice_id.startswith("em_"):
        language, gender = "Spanish", "Male"

    short_name = voice_id.split("_")[-1].capitalize()

    processed_voices.append(
        {
            "Voice_ID": voice_id,
            "Name": short_name,
            "Language": language,
            "Gender": gender,
        }
    )

self_updating_df = pd.DataFrame(processed_voices)

# ==========================================
# PART 2: Scrape live grades from VOICES.md
# ==========================================
url = "https://huggingface.co/hexgrad/Kokoro-82M/raw/main/VOICES.md"
response = requests.get(url)
md_content = response.text

lines = md_content.split("\n")
table_data = []

for line in lines:
    # Adjusted regex to handle the markdown pipe symbols at the start of the row
    # It strips whitespace and checks if the column content starts with a voice ID format
    clean_line = line.strip()
    if clean_line.startswith("|"):
        parts = [p.strip() for p in clean_line.split("|") if p.strip() != ""]
        if parts and re.match(r"^[a-z]{2}_[a-z]+", parts[0]):
            # Verify we have enough columns to extract attributes safely
            if len(parts) >= 4:
                table_data.append(
                    {
                        "Voice_ID": parts[0],
                        "Traits": parts[1] if len(parts) > 1 else "",
                        "Target_Quality": parts[2] if len(parts) > 2 else "",
                        "Training_Duration": parts[3] if len(parts) > 3 else "",
                        "Overall_Grade": parts[4] if len(parts) > 4 else "Unrated",
                    }
                )

# Create the DataFrame from scraped data
if table_data:
    grades_df = pd.DataFrame(table_data)
else:
    # Fallback to empty DataFrame with explicit column headers if parsing failed
    print("Warning: Web scraper could not find data rows. Creating an empty fallback table.")
    grades_df = pd.DataFrame(columns=["Voice_ID", "Traits", "Target_Quality", "Training_Duration", "Overall_Grade"])

# ==========================================
# PART 3: Force alignment and merge safely
# ==========================================
# Strip trailing whitespaces from column headers safely
self_updating_df.columns = self_updating_df.columns.str.strip()
grades_df.columns = grades_df.columns.str.strip()

# Complete the left merge seamlessly
full_registry_df = self_updating_df.merge(grades_df, on="Voice_ID", how="left")

# Sort cleanly by grade tiers (highest performance options first)
full_registry_df.sort_values(
    by=["Language", "Overall_Grade", "Name"],
    ascending=[True, True, True],
    inplace=True,
)
full_registry_df.reset_index(drop=True, inplace=True)

print(f"Successfully synced {len(grades_df)} graded rows live from Hugging Face!")
full_registry_df.head(5)

Successfully synced 47 graded rows live from Hugging Face!


,Voice_ID,Name,Language,Gender,Traits,Target_Quality,Training_Duration,Overall_Grade
0,af_bella,Bella,American English,Female,🚺🔥,**A**,**HH hours**,**A-**
1,af_nicole,Nicole,American English,Female,🚺🎧,B,**HH hours**,B-
2,af_alloy,Alloy,American English,Female,🚺,B,MM minutes,C
3,af_nova,Nova,American English,Female,🚺,B,MM minutes,C
4,af_aoede,Aoede,American English,Female,🚺,B,H hours,C+


In [10]:
# Test voices from full_registery_df

# 1. Define a standard test passage
test_text = "The quick brown fox jumps over the lazy dog. This is a test of the local Kokoro speech synthesis engine, currently running on hardware acceleration."

def play_voice_live(voice_id, pipeline):
    print(f"Streaming voice live: {voice_id}...")
    
    # 1. Generate the audio
    generator = pipeline(test_text, voice=voice_id, speed=1, split_pattern=r'\n+')
    
    # 2. Extract the audio array
    for _, _, audio in generator:
        # 3. Create an in-memory buffer (BytesIO)
        # We act as if we are writing a wav, but it stays in your RAM
        buffer = io.BytesIO()
        sf.write(buffer, audio, 24000, format='WAV')
        buffer.seek(0) # Reset the pointer to the start of the buffer
        
        # 4. Play directly from the memory buffer
        pygame.mixer.init()
        sound = pygame.mixer.Sound(buffer)
        sound.play()
        
        print("Audio playing directly from RAM.")
        return

# 2. Pick a specific voice ID directly from your Registry DataFrame
# Change "chosen_voice_id" to any voice ID string you want to test from full_registry_df
chosen_voice_id = "af_bella"

# Verify the voice exists in your dataframe before passing it to the pipeline
if chosen_voice_id in full_registry_df['Voice_ID'].values:
    # 3. Run the live in-memory generation loop
    play_voice_live(chosen_voice_id, pipeline)
else:
    print(f"⚠️ Voice ID '{chosen_voice_id}' was not found in your current self_updating dataframe layout.")

Streaming voice live: af_bella...
Audio playing directly from RAM.
